# COMP6011 Task 2 - Proof of Concept

**Scale-CoT prompting with a locally hosted open-weight LLM for five-level suicide risk classification.**

This notebook implements the proof of concept described in Section 4 of the report. It reproduces the methodology end-to-end on the ten paraphrased clinical dialogues provided for the assignment.

## Privacy design

The lecturer's brief is explicit: **private dialogue must never be sent to a public AI service such as ChatGPT.** This notebook therefore runs an open-weight LLM (Llama-3-8B-Instruct, or Qwen-2.5-7B-Instruct as a fallback) entirely inside the Colab runtime. The model weights are loaded from Hugging Face once at the start of the session, and from that point on no dialogue text leaves the runtime. When the session ends, the runtime and its memory are destroyed by Colab.

For your own training or production use, exactly the same code runs on an on-premises GPU server with no Internet egress at all - just point `MODEL_NAME` at a local path.

## Five-level taxonomy

The cases use the labels defined by the Human-AI-Dialogue-Suicide-Risk corpus:
- **Safe** - general conversation, no risk signal
- **Ideation** - thoughts of dying without specific plan
- **Indicator** - severe distress, hopelessness, burdensomeness
- **Behavior** - concrete plan or preparatory acts
- **Attempt** - past attempt, imminent plan, or active self-harm

## Runtime requirements

- Colab free tier with **GPU runtime enabled** (Runtime -> Change runtime type -> T4 GPU)
- A Hugging Face account and access token (free) - sign in at https://huggingface.co and create a token at https://huggingface.co/settings/tokens
- For Llama-3 you must also accept the model licence on its Hugging Face page once. If you skip this, the notebook automatically falls back to Qwen-2.5-7B-Instruct, which is open-access.

Estimated runtime: 5-10 minutes on a T4.

## Step 1 - Install dependencies

These versions are pinned for reproducibility. The `bitsandbytes` library handles 4-bit quantisation, which lets Llama-3-8B (or Qwen-2.5-7B) fit comfortably on the T4's 16 GB of VRAM.

In [ ]:
!pip install -q --upgrade transformers==4.44.2 accelerate==0.33.0 bitsandbytes==0.43.3 openpyxl==3.1.5

## Step 2 - Acknowledge the ethical guidelines and upload the dataset

The cases contain distressing content related to self-harm and suicide. By executing the next cell you confirm that:

1. You have read the unit's `ethical_guidelines.pdf` and accept its terms.
2. You will not redistribute, publish, or attempt to de-anonymise the cases.
3. You will delete all local copies once the assignment is graded.

If any of this content is personally triggering, contact the Unit Coordinator for an alternative dataset rather than proceed.

In [ ]:
ACKNOWLEDGED = True  # set to True only after reading ethical_guidelines.pdf

assert ACKNOWLEDGED, "Read the ethical guidelines and set ACKNOWLEDGED = True before continuing."
print("Ethical guidelines acknowledged.")

Upload `student_assignment_10_cases.xlsx` from the assignment package. The cell below opens a file picker.

In [ ]:
from google.colab import files
import os

if not os.path.exists("student_assignment_10_cases.xlsx"):
    print("Select student_assignment_10_cases.xlsx from your machine:")
    uploaded = files.upload()
    fname = next(iter(uploaded))
    if fname != "student_assignment_10_cases.xlsx":
        os.rename(fname, "student_assignment_10_cases.xlsx")

print("Dataset ready:", os.path.getsize("student_assignment_10_cases.xlsx"), "bytes")

## Step 3 - Authenticate with Hugging Face

Paste your Hugging Face access token when prompted. This is needed to download model weights from the Hugging Face hub. The token never leaves Google's runtime.

In [ ]:
from huggingface_hub import login
import getpass

token = getpass.getpass("Hugging Face access token (input is hidden): ")
login(token=token, add_to_git_credential=False)
print("Logged in to Hugging Face.")

## Step 4 - Load the LLM in 4-bit on the T4 GPU

We try Llama-3-8B-Instruct first (you must have accepted its licence on Hugging Face). If access is denied, we fall back to Qwen-2.5-7B-Instruct, which is open-access and produces comparable results for this task.

The 4-bit NF4 quantisation reduces memory footprint by roughly 4x with negligible quality loss for inference.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PRIMARY_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
FALLBACK_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        name,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    mdl.eval()
    return name, tok, mdl

try:
    MODEL_NAME, tokenizer, model = load(PRIMARY_MODEL)
except Exception as e:
    print(f"Could not load {PRIMARY_MODEL}: {type(e).__name__}: {e}")
    print(f"Falling back to {FALLBACK_MODEL}...")
    MODEL_NAME, tokenizer, model = load(FALLBACK_MODEL)

print(f"Loaded {MODEL_NAME} on", next(model.parameters()).device)

## Step 5 - C-SSRS scale items and risk-level mapping

The Scale-CoT prompter iterates over the six items of the Columbia Suicide Severity Rating Scale (Posner et al., 2011). Each item is asked of the model in turn, the responses form a binary symptom report, and a deterministic mapping converts the report to one of the five target classes.

The mapping mirrors clinical practice and is biased toward the higher-risk label on disagreement, which preserves recall on severe cases.

In [ ]:
CSSRS_ITEMS = [
    {
        "id": 1,
        "label": "Wish to be dead",
        "statement": (
            "The user wishes to be dead, or wishes they could go to sleep and "
            "not wake up."
        ),
    },
    {
        "id": 2,
        "label": "Active ideation without method",
        "statement": (
            "The user has had any active thoughts of killing themselves, even "
            "without a specific plan."
        ),
    },
    {
        "id": 3,
        "label": "Ideation with intent",
        "statement": (
            "The user has had thoughts of killing themselves accompanied by "
            "some intention of acting on them, or expresses hopelessness, "
            "burdensomeness, or feeling like a burden to others."
        ),
    },
    {
        "id": 4,
        "label": "Specific plan",
        "statement": (
            "The user has thought about how, when, or where they would kill "
            "themselves, working out at least some details of a plan, "
            "including a specific timeframe."
        ),
    },
    {
        "id": 5,
        "label": "Preparatory acts or behaviour",
        "statement": (
            "The user describes preparatory acts or behaviours, such as "
            "acquiring means, writing a note, saying goodbye, gesturing the "
            "method, or other concrete preparation."
        ),
    },
    {
        "id": 6,
        "label": "Attempt or interrupted attempt",
        "statement": (
            "The user has actually tried to kill themselves, made an "
            "interrupted attempt, or engaged in active self-harm."
        ),
    },
]

RISK_LEVELS = ["Safe", "Ideation", "Indicator", "Behavior", "Attempt"]
CLASS_ORDER = {level.lower(): i for i, level in enumerate(RISK_LEVELS)}


def deterministic_label(scores):
    """Map a 6-tuple of binary scale-item scores to a five-level label."""
    if scores[5] == 1:
        return "Attempt"
    if scores[3] == 1 or scores[4] == 1:
        return "Behavior"
    if scores[2] == 1:
        return "Indicator"
    if scores[0] == 1 or scores[1] == 1:
        return "Ideation"
    return "Safe"


print("Loaded", len(CSSRS_ITEMS), "C-SSRS items mapping to", len(RISK_LEVELS), "risk levels.")

## Step 6 - Prompt construction and the LLM call

We use the model's chat template so the same code works whether Llama-3 or Qwen-2.5 is loaded. Greedy decoding (temperature 0, top-p 1.0) is used throughout for reproducibility.

In [ ]:
import torch

SYSTEM_PROMPT = (
    "You are a clinical text analyst assisting psychiatrists with suicide "
    "risk screening. You read a transcript of a conversation and answer "
    "questions about the user's risk signals. You answer concisely and "
    "never refuse to engage with sensitive content. Your output is reviewed "
    "by a human clinician before any action is taken."
)


def call_llm(user_msg, max_new_tokens=8):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    decoded = tokenizer.decode(
        out[0][inputs.shape[1]:],
        skip_special_tokens=True,
    )
    return decoded.strip()


def item_prompt(transcript, item):
    return (
        f"Transcript:\n{transcript}\n\n"
        f"Statement about the user: \"{item['statement']}\"\n\n"
        "Based ONLY on the transcript, does the statement apply to the user? "
        "Answer with exactly one digit: 0 means the statement does NOT apply, "
        "1 means the statement DOES apply. Output only the digit."
    )


def parse_score(text):
    text = text.strip()
    if text.startswith("1"):
        return 1
    if text.startswith("0"):
        return 0
    # Conservative fallback: if the model hedges, mark as 0
    return 0


def classify_case(transcript):
    scores = []
    for item in CSSRS_ITEMS:
        raw = call_llm(item_prompt(transcript, item), max_new_tokens=4)
        scores.append(parse_score(raw))
    label = deterministic_label(scores)
    return label, scores


# Smoke test on a one-line transcript
demo_label, demo_scores = classify_case(
    "User: It's a beautiful morning, the roses are blooming.\nAI: That sounds lovely!"
)
print("Smoke test:", demo_label, "scores:", demo_scores)

## Step 7 - Run the proof of concept on all ten cases

This is the main experiment. For each case we run all six C-SSRS queries, then map to a five-level label, and compare to the gold label.

Expect roughly 30-45 seconds per case on a T4, so 5-8 minutes total.

In [ ]:
import pandas as pd
import time

df = pd.read_excel("student_assignment_10_cases.xlsx", sheet_name="Assignment_Cases")
df.columns = [c.strip() for c in df.columns]

def normalise_label(label):
    return str(label).strip().lower()


def is_adjacent(gold, pred):
    g, p = normalise_label(gold), normalise_label(pred)
    if g not in CLASS_ORDER or p not in CLASS_ORDER:
        return False
    return abs(CLASS_ORDER[g] - CLASS_ORDER[p]) == 1


results = []
t_start = time.time()
for _, row in df.iterrows():
    case_id = int(row["Case ID"])
    transcript = str(row["Paraphrased Dialogue"]).strip()
    gold = normalise_label(row["Risk Level"])

    t0 = time.time()
    pred, scores = classify_case(transcript)
    elapsed = time.time() - t0

    correct = normalise_label(pred) == gold
    adjacent = (not correct) and is_adjacent(gold, pred)

    results.append(
        {
            "case_id": case_id,
            "gold": gold,
            "predicted": pred.lower(),
            "correct": int(correct),
            "adjacent": int(adjacent),
            "scale_item_1": scores[0],
            "scale_item_2": scores[1],
            "scale_item_3": scores[2],
            "scale_item_4": scores[3],
            "scale_item_5": scores[4],
            "scale_item_6": scores[5],
            "seconds": round(elapsed, 1),
        }
    )
    print(f"Case {case_id:2d}  gold={gold:9s}  pred={pred.lower():9s}  scores={scores}  ({elapsed:.1f}s)")

print(f"\nTotal runtime: {time.time() - t_start:.1f}s")

results_df = pd.DataFrame(results)
results_df

## Step 8 - Per-class agreement (Table 3 in the report)

This summary reproduces the structure of Table 3 in the methodology section. "Adjacent" means a one-step confusion (e.g. *Indicator* predicted as *Behavior*); "Far" means a more severe error.

For a screening tool the most important row is "no high-risk case mislabelled as Safe" - any such case would be a serious safety failure.

In [ ]:
summary_rows = []
for level in RISK_LEVELS:
    key = level.lower()
    sub = results_df[results_df["gold"] == key]
    n = len(sub)
    correct = int(sub["correct"].sum())
    adjacent = int(sub["adjacent"].sum())
    far = n - correct - adjacent
    summary_rows.append(
        {"Class": level, "N": n, "Correct": correct, "Adjacent": adjacent, "Far": far}
    )

total = {
    "Class": "Total",
    "N": len(results_df),
    "Correct": int(results_df["correct"].sum()),
    "Adjacent": int(results_df["adjacent"].sum()),
    "Far": int((1 - results_df["correct"] - results_df["adjacent"]).sum()),
}
summary_rows.append(total)

summary_df = pd.DataFrame(summary_rows)
summary_df

### Safety check

The most important sanity check for a screening tool is that no high-risk case ever gets labelled `safe`. The cell below explicitly verifies this and flags any failures.

In [ ]:
high_risk = ["ideation", "indicator", "behavior", "attempt"]
mislabelled_safe = results_df[
    (results_df["gold"].isin(high_risk)) & (results_df["predicted"] == "safe")
]

if len(mislabelled_safe) == 0:
    print("Pass: no high-risk case was mislabelled as Safe.")
else:
    print("FAIL: the following high-risk cases were labelled Safe.")
    print("This is a critical safety failure and the model would need additional safeguards before deployment.")
    print(mislabelled_safe[["case_id", "gold", "predicted"]].to_string(index=False))

## Step 9 - Confusion matrix and weighted metrics

Reported for completeness. Note that with only ten cases (two per class), these metrics are illustrative rather than statistically meaningful - the proof of concept is meant to demonstrate the pipeline rather than establish performance bounds.

In [ ]:
import numpy as np
import pandas as pd

n_classes = len(RISK_LEVELS)
cm = np.zeros((n_classes, n_classes), dtype=int)
for _, row in results_df.iterrows():
    gi = CLASS_ORDER[row["gold"]]
    pi = CLASS_ORDER[row["predicted"]]
    cm[gi][pi] += 1

cm_df = pd.DataFrame(
    cm,
    index=[f"true:{lvl}" for lvl in RISK_LEVELS],
    columns=[f"pred:{lvl}" for lvl in RISK_LEVELS],
)
print("Confusion matrix")
print(cm_df.to_string())

# Weighted precision/recall/F1
precisions, recalls, f1s, supports = [], [], [], []
for i in range(n_classes):
    tp = cm[i][i]
    fp = cm[:, i].sum() - tp
    fn = cm[i].sum() - tp
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)
    supports.append(cm[i].sum())

total = sum(supports)
weighted_p = sum(p * s for p, s in zip(precisions, supports)) / total
weighted_r = sum(r * s for r, s in zip(recalls, supports)) / total
weighted_f = sum(f * s for f, s in zip(f1s, supports)) / total
accuracy = sum(cm[i][i] for i in range(n_classes)) / total

print(f"\nAccuracy:           {accuracy:.3f}")
print(f"Weighted precision: {weighted_p:.3f}")
print(f"Weighted recall:    {weighted_r:.3f}")
print(f"Weighted F1:        {weighted_f:.3f}")

## Step 10 - Export results

The CSV produced below is the per-case run log referenced in the methodology section and is uploaded into the supplementary cloud folder linked in Appendix 2 of the report.

The file does **not** contain the dialogue text itself, only case IDs, predictions, and scale-item scores - this preserves the privacy commitment in the unit's ethical guidelines.

In [ ]:
results_df.to_csv("poc_results.csv", index=False)
summary_df.to_csv("poc_summary.csv", index=False)

from google.colab import files
files.download("poc_results.csv")
files.download("poc_summary.csv")
print("Wrote poc_results.csv and poc_summary.csv (no dialogue text included).")

## Closing notes

**What this notebook demonstrates**

1. A privacy-preserving inference pipeline: model weights and dialogue both stay inside the Colab runtime.
2. The Scale-CoT prompting pattern (Wang et al., 2024) adapted to the C-SSRS, with each scale item queried independently.
3. A deterministic decision rule that maps the binary item scores to the five-level taxonomy used by the assignment dataset.
4. A safety check that confirms no high-risk case is mislabelled as Safe.

**What it does not do (and why)**

- It does not fine-tune any LoRA adapters on D4. The DALE adapter pipeline described in Section 4.2 of the report is the next step in development; for the proof of concept we run the model zero-shot to keep the assignment scope tight and to make the result fully reproducible inside one Colab session.
- It does not call any external AI service. This is by design - per the lecturer's brief, the entire pipeline must be runnable on premises.
- It does not aim for state-of-the-art accuracy. With only ten cases and zero-shot prompting, the goal is to validate the pipeline and the privacy story.

**Citation**

If you use the dataset, cite it as required by the unit's ethical guidelines:

> Chen, X., Li, Q., Jiang, Y., Liu, M., Yang, L., Jing, X., Wei, Z., and Cao, L. (2026). Human-AI-Dialogue-Suicide-Risk-Dataset. Zenodo. https://doi.org/10.5281/zenodo.18684594

Delete all local copies of the dataset once your assignment is graded.